You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
from utils import load_env
load_env()

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [17]:
planner = Agent(
    role="Content Planner",
    goal="围绕 {topic} 策划引人入胜且事实准确的内容",
    backstory="你正在为一篇关于主题 {topic} 的博客文章做策划。"
              "你收集有助于读者学习新知识、"
              "并做出明智决策的信息。"
              "你的工作将作为内容撰写者"
              "撰写该主题文章的基础。",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [18]:
writer = Agent(
    role="Content Writer",
    goal="就主题 {topic} 撰写有见地且事实准确的评论文章",
    backstory="你正在撰写一篇关于主题 {topic} 的新评论文章。"
              "你的写作基于内容策划者的工作，"
              "对方提供了大纲以及该主题的相关背景。"
              "你遵循大纲中的主要目标与写作方向，"
              "这些内容由内容策划者提供。"
              "你还提供客观、中立的见解，"
              "并用内容策划者提供的信息加以支撑。"
              "在评论文章中，当陈述属于观点而非客观事实时，"
              "你会明确予以说明。",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [19]:
editor = Agent(
    role="Editor",
    goal="编辑给定的博客文章，使其符合组织的写作风格",
    backstory="你是一名编辑，会收到内容撰写者提交的博客文章。"
              "你的目标是审阅该博客文章，"
              "确保其符合新闻写作的最佳实践，"
              "在表达观点或论断时提供平衡的视角，"
              "并尽可能避免涉及重大争议性话题或观点。",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [20]:
plan = Task(
    description=(
        "1. 优先梳理 {topic} 领域的最新趋势、主要参与方和重要新闻。\n"
        "2. 明确目标受众，并考虑其兴趣点与痛点。\n"
        "3. 制定详细的内容大纲，包括引言、要点和行动号召。\n"
        "4. 包含 SEO 关键词以及相关的数据或资料来源。"
    ),
    expected_output="一份全面的内容策划文档，"
        "包含大纲、受众分析、SEO 关键词和资源。",
    agent=planner,
)

### Task: Write

In [21]:
write = Task(
    description=(
        "1. 根据内容策划撰写关于 {topic} 的引人入胜的博客文章。\n"
        "2. 自然地融入 SEO 关键词。\n"
        "3. 各章节/小标题命名得当且富有吸引力。\n"
        "4. 确保文章结构清晰：引人入胜的开头、有见地的正文、总结性的结尾。\n"
        "5. 校对语法错误，并确保与品牌调性一致。\n"
    ),
    expected_output="一篇撰写精良、可直接发布的 Markdown 格式博客文章，"
        "每个章节应包含 2 到 3 个段落。",
    agent=writer,
)

### Task: Edit

In [22]:
edit = Task(
    description=("校对给定的博客文章，检查语法错误"
                 "并确保与品牌调性一致。"),
    expected_output="一篇撰写精良、可直接发布的 Markdown 格式博客文章，"
                    "每个章节应包含 2 到 3 个段落。",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` enables detailed logs (newer crewAI uses bool, not `0/1/2`). 

In [23]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

**Jupyter**: Use `await crew.kickoff_async(...)` instead of `crew.kickoff()` — notebooks already run an asyncio event loop.

In [24]:
result = await crew.kickoff_async(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c0dbf109-16c3-403a-a880-6accdc874581                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. 优先梳理 Artificial Intelligence 领域的最新趋势、主要参与方和重要新闻。                               │
│  2. 明确目标受众，并考虑其兴趣点与痛点。                                                                        │
│  3. 制定详细的内容大纲，包括引言、要点和行动号召。                                                              │
│  4. 包含 SEO 关键词以及相关的数据或资料来源。                                                                   │
│  ID: e1bbcb0a-4a9f-4230-91f5-d79879c79dce                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. 优先梳理 Artificial Intelligence 领域的最新趋势、主要参与方和重要新闻。                               │
│  2. 明确目标受众，并考虑其兴趣点与痛点。                                                                        │
│  3. 制定详细的内容大纲，包括引言、要点和行动号召。                                                              │
│  4. 包含 SEO 关键词以及相关的数据或资料来源。                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 博客文章内容策划文档：人工智能（Artificial Intelligence）全景指南                                            │
│                                                                                                                 │
│  **项目名称：** 2024 人工智能战略指南：从认知到行动                                                             │
│  **策划人：** Content Planner                                                                                   │
│  **日期：** 2024 年 5 月                                                                                        │
│  **主题：** Artificial Intelligence (AI)                                                                        │
│  **主要目标：** 提供准确、前沿的 AI 知识，消除读者焦虑，并提供可执行的决策建议。                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 第一部分：行业现状与趋势梳理 (Industry Landscape)                                                           │
│                                                                                                                 │
│  在撰写文章前，必须明确当前 AI 领域的核心动态，以确保内容的时效性和准确性。                                     │
│                                                                                                                 │
│  ### 1. 最新技术趋势 (Latest Trends)                                                                            │
│  *   **生成式 AI 的平民化 (Democratization of GenAI):**                                                         │
│  从单纯的文本生成向多模态（图像、视频、音频）融合转变。例如，Sora 等视频生成模型的发布标志着理解力的提升。      │
│  *   **AI Agent（智能体）的兴起:** AI 不再只是对话工具，而是能够自主规划、调用工具并完成任务的智能体（如        │
│  AutoGPT, LangChain 生态）。                                                                                    │
│  *   **边缘 AI (Edge AI):** 为了隐私和延迟，算力正从云端向端侧设备（手机、PC、汽车）转移（如 Apple 的端侧 Siri  │
│  升级，NVIDIA 的 Jetson 系列）。                                                                                │
│  *   **监管与伦理 (Regulation & Ethics):** 欧盟《AI 法案》(EU AI Act) 的实施，以及各国对                        │
│  Deepfake（深度伪造）、数据版权和数据安全的关注成为焦点。                                                       │
│                                                                                                                 │
│  ### 2. 主要参与方 (Key Players)                                                                                │
│  *   **科技巨头 (Hyperscalers):** OpenAI (ChatGPT), Google (Gemini), Microsoft (Copilot), Meta (Llama           │
│  开源模型), Amazon。                                                                                            │
│  *   **中国领军企业:** 百度 (文心一言), 阿里巴巴 (通义千问), 腾讯 (混元), 科大讯飞。                            │
│  *   **初创公司与垂直领域:** Anthropic (Claude), Stability AI (Midjourney 竞品),                                │
│  以及专注于法律、医疗、金融垂直行业的 AI 独角兽。                                                               │
│                                                                                                                 │
│  ### 3. 重要新闻与事件 (Important News)                                                                         │
│  *   **全球 AI 安全峰会 (Global AI Safety Summit):** 讨论存在性风险与对齐问题。                                 │
│  *   **开源 vs. 闭源之争:** Meta

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 2024 人工智能生存指南：趋势解析、职业影响与实战策略                                                          │
│  ## 告别焦虑，掌握驾驭 AI 的核心框架                                                                            │
│                                                                                                                 │
│  根据麦肯锡全球研究院的最新报告，**生成式 AI** 每年可能为全球 GDP                                               │
│  贡献数万亿美元的价值。这不再是一个遥远的预测，而是正在发生的现实。就在过去的一年里，我们见证了从单纯的文本聊   │
│  天机器人到能够理解视频、生成代码的智能体的技术迭代。然而，技术的爆炸式增长也带来了前所未有的信息过载和不确定   │
│  性。对于许多职场人士和企业主而言，面对层出不穷的新技术，最普遍的感受并非兴奋，而是担忧自己是否会被时代抛下。   │
│                                                                                                                 │
│  本文旨在拨开技术炒作的迷雾，为您提供一份基于事实的**人工智能趋势 2024** 全景指南。我们将深入探讨当前 AI        │
│  的核心动态，分析其对职业生态的真实影响，并为不同层级的读者提供可执行的决策建议。我们的目标不是制造焦虑，而是   │
│  通过客观的数据和行业洞察，帮助您区分哪些是短暂的泡沫，哪些是长期的价值，从而在变革中掌握主动权。               │
│                                                                                                                 │
│  无论您是寻求技能转型的职场专业人士，还是寻找降本增效方案的中小企业主，这场变革都不再是一个“选择题”，而是一个   │
│  必须面对的“必答题”。接下来，我们将通过六个关键维度，构建一个从认知到行动的完整框架，助您从容应对这一轮技术浪   │
│  潮。                                                                                                           │
│                                                                                                                 │
│  ## 核心趋势：超越对话的行动智能                                                                                │
│                                                                                                                 │
│  当我们谈论**生成式 AI 应用**时，公众的目光往往集中在 ChatGPT                                                   │
│  等对话式工具上，但真正的行业突破正发生在“行动”层面。目前，**AI Agent（智能体）**                               │
│  正在兴起，它们不再仅仅是被动回答问题的助手，而是能够自主规划任务、调用外部工具并执行复杂工作流程的系统。例如   │
│  ，AutoGPT 和 LangChain 生态的出现，意味着 AI                                                                   │
│  可以独立完成预订行程、编写并部署代码或进行数据分析，这标志着人机交互从“问答模式”向“任务委托模式”的显著转变。   │
│                                                                                                                 │
│  与此同时，多模态融合成为了不可忽视的技术风向标。以 Sora 为代表的视频生成模型展示了 AI                          │
│  对物理世界理解的显著提升，将文本、图像、音频和视频的处理能力统一化。这种能力的整合意味着未来的交互方式将更加   │
│  自然和直观，用户无需专门学习复杂的指令即可与机器沟通。据斯坦福大学人本人工智能研究所（HAI）发布的《AI Index    │
│  Report》显示，多模态模型的性能曲线在过去一年内呈现指数级增长，预示着跨媒体内容创作将成为主流应用场景。         │
│                                                                                                                 │
│  然而，技术的狂奔也引来了监管的收紧与伦理的审视。随着欧盟《AI                                                   │
│  法案》的推进以及各国对深度伪造（Deepfake）和数据安全的关注，合规性已成为技术发展的重要约束条件。**人工智能趋   │
│  势 2024**                                                                                                      │
│  的另一大特点是隐私计算的强化，算力正逐渐从云端向边缘设备（如智能手机、PC）转移。这不仅是为了降低延迟，更是为   │
│  了在源头上保护用户数据，确保在享受便捷的同时不牺牲隐私安全。                                                   │
│                                                                                                                 │
│  ## 竞争版图：巨头博弈与生态分层                                                                                │
│                                                                                                                 │
│  当前的 AI 生态系统呈现出明显的分层结构，且主要参与者之间的竞争愈发激烈。在通用模型层，以                       │
│  OpenAI、Google、Microsoft                                                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 校对给定的博客文章，检查语法错误并确保与品牌调性一致。                                                   │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c0dbf109-16c3-403a-a880-6accdc874581                                                                       │
│  Final Output: # 2024 人工智能生存指南：趋势解析、职业影响与实战策略                                            │
│  ## 告别焦虑，掌握驾驭 AI 的核心框架                                                                            │
│                                                                                                                 │
│  根据麦肯锡全球研究院的最新报告，**生成式 AI** 每年可能为全球 GDP                                               │
│  贡献数万亿美元的价值。这不再是一个遥远的预测，而是正在发生的现实。就在过去的一年里，我们见证了从单纯的文本聊   │
│  天机器人到能够理解视频、生成代码的智能体的技术迭代。然而，技术的爆炸式增长也带来了前所未有的信息过载和不确定   │
│  性。对于许多职场人士和企业主而言，面对层出不穷的新技术，最普遍的感受并非兴奋，而是担忧自己是否会被时代抛下。   │
│                                                                                                                 │
│  本文旨在拨开技术炒作的迷雾，为您提供一份基于事实的**人工智能趋势 2024** 全景指南。我们将深入探讨当前 AI        │
│  的核心动态，分析其对职业生态的真实影响，并为不同层级的读者提供可执行的决策建议。我们的目标不是制造焦虑，而是   │
│  通过客观的数据和行业洞察，帮助您区分哪些是短暂的泡沫，哪些是长期的价值，从而在变革中掌握主动权。               │
│                                                                                                                 │
│  无论您是寻求技能转型的职场专业人士，还是寻找降本增效方案的中小企业主，这场变革都不再是一个“选择题”，而是一个   │
│  必须面对的“必答题”。接下来，我们将通过六个关键维度，构建一个从认知到行动的完整框架，助您从容应对这一轮技术浪   │
│  潮。                                                                                                           │
│                                                                                                                 │
│  ## 核心趋势：超越对话的行动智能                                                                                │
│                                                                                                                 │
│  当我们谈论**生成式 AI 应用**时，公众的目光往往集中在 ChatGPT                                                   │
│  等对话式工具上，但真正的行业突破正发生在“行动”层面。目前，**AI Agent（智能体）**                               │
│  正在兴起，它们不再仅仅是被动回答问题的助手，而是能够自主规划任务、调用外部工具并执行复杂工作流程的系统。例如   │
│  ，AutoGPT 和 LangChain 生态的出现，意味着 AI                                                                   │
│  可以独立完成预订行程、编写并部署代码或进行数据分析，这标志着人机交互从“问答模式”向“任务委托模式”的显著转变。   │
│                                                                                                                 │
│  与此同时，多模态融合成为了不可忽视的技术风向标。以 Sora 为代表的视频生成模型展示了 AI                          │
│  对物理世界理解的显著提升，将文本、图像、音频和视频的处理能力统一化。这种能力的整合意味着未来的交互方式将更加   │
│  自然和直观，用户无需专门学习复杂的指令即可与机器沟通。据斯坦福大学人本人工智能研究所（HAI）发布的《AI Index    │
│  Report》显示，多模态模型的性能曲线在过去一年内呈现指数级增长，预示着跨媒体内容创作将成为主流应用场景。         │
│                                                                                                                 │
│  然而，技术的狂奔也引来了监管的收紧与伦理的审视。随着欧盟《AI                                                   │
│  法案》的推进以及各国对深度伪造（Deepfake）和数据安全的关注，合规性已成为技术发展的重要约束条件。**人工智能趋   │
│  势 2024**                                                                                                      │
│  的另一大特点是隐私计算的强化，算力正逐渐从云端向边缘设备（如智能手机、PC）转移。这不仅是为了降低延迟，更是为   │
│  了在源头上保护用户数据，确保在享受便捷的同时不牺牲隐私安全。                                                   │
│                                                                                                                 │
│  ## 竞争版图：巨头博弈与生态分层                                                                                │
│                                                                                                                 │
│  当前的 AI 生态系统呈现出明显的分层结构，且主要参与者之间的竞争愈发激烈。在通用模型层，以                       │
│  OpenAI、Google、Microsoft                                                          

- Display the results of your execution as markdown in the notebook.

In [26]:
from IPython.display import Markdown
Markdown(result.raw)

# 2024 人工智能生存指南：趋势解析、职业影响与实战策略
## 告别焦虑，掌握驾驭 AI 的核心框架

根据麦肯锡全球研究院的最新报告，**生成式 AI** 每年可能为全球 GDP 贡献数万亿美元的价值。这不再是一个遥远的预测，而是正在发生的现实。就在过去的一年里，我们见证了从单纯的文本聊天机器人到能够理解视频、生成代码的智能体的技术迭代。然而，技术的爆炸式增长也带来了前所未有的信息过载和不确定性。对于许多职场人士和企业主而言，面对层出不穷的新技术，最普遍的感受并非兴奋，而是担忧自己是否会被时代抛下。

本文旨在拨开技术炒作的迷雾，为您提供一份基于事实的**人工智能趋势 2024** 全景指南。我们将深入探讨当前 AI 的核心动态，分析其对职业生态的真实影响，并为不同层级的读者提供可执行的决策建议。我们的目标不是制造焦虑，而是通过客观的数据和行业洞察，帮助您区分哪些是短暂的泡沫，哪些是长期的价值，从而在变革中掌握主动权。

无论您是寻求技能转型的职场专业人士，还是寻找降本增效方案的中小企业主，这场变革都不再是一个“选择题”，而是一个必须面对的“必答题”。接下来，我们将通过六个关键维度，构建一个从认知到行动的完整框架，助您从容应对这一轮技术浪潮。

## 核心趋势：超越对话的行动智能

当我们谈论**生成式 AI 应用**时，公众的目光往往集中在 ChatGPT 等对话式工具上，但真正的行业突破正发生在“行动”层面。目前，**AI Agent（智能体）** 正在兴起，它们不再仅仅是被动回答问题的助手，而是能够自主规划任务、调用外部工具并执行复杂工作流程的系统。例如，AutoGPT 和 LangChain 生态的出现，意味着 AI 可以独立完成预订行程、编写并部署代码或进行数据分析，这标志着人机交互从“问答模式”向“任务委托模式”的显著转变。

与此同时，多模态融合成为了不可忽视的技术风向标。以 Sora 为代表的视频生成模型展示了 AI 对物理世界理解的显著提升，将文本、图像、音频和视频的处理能力统一化。这种能力的整合意味着未来的交互方式将更加自然和直观，用户无需专门学习复杂的指令即可与机器沟通。据斯坦福大学人本人工智能研究所（HAI）发布的《AI Index Report》显示，多模态模型的性能曲线在过去一年内呈现指数级增长，预示着跨媒体内容创作将成为主流应用场景。

然而，技术的狂奔也引来了监管的收紧与伦理的审视。随着欧盟《AI 法案》的推进以及各国对深度伪造（Deepfake）和数据安全的关注，合规性已成为技术发展的重要约束条件。**人工智能趋势 2024** 的另一大特点是隐私计算的强化，算力正逐渐从云端向边缘设备（如智能手机、PC）转移。这不仅是为了降低延迟，更是为了在源头上保护用户数据，确保在享受便捷的同时不牺牲隐私安全。

## 竞争版图：巨头博弈与生态分层

当前的 AI 生态系统呈现出明显的分层结构，且主要参与者之间的竞争愈发激烈。在通用模型层，以 OpenAI、Google、Microsoft 为代表的科技巨头占据了市场主导权，它们依托强大的资本和数据优势不断迭代大模型版本。与此同时，中国领军企业如百度的文心一言、阿里巴巴的通义千问以及腾讯混元也在本地化服务和中文场景优化上展现出强劲竞争力。对于普通用户和企业而言，选择何种基座模型取决于具体的业务需求、数据合规要求以及成本预算。

除了闭源巨头的垄断，开源生态的力量也不容小觑。Meta 推出的 Llama 系列模型推动了**大模型落地**过程中的开源进程，使得中小企业和个人开发者能够以更低的成本微调属于自己的专用模型。这种“开源 vs. 闭源”的争论实际上反映了两种不同的发展路径：一方追求极致的性能与安全性控制，另一方则强调社区的协作创新与灵活性。我们认为，未来市场更有可能形成混合模式，即核心能力由大厂提供 API，而个性化应用则由开源社区定制。

基础设施层的变动同样影响着整体成本结构。GPU 算力的短缺曾一度成为制约发展的瓶颈，但随着 NVIDIA 等厂商产能的提升及云服务商的竞争，单位计算成本有望逐步下降。不过，这也意味着 AI 服务的价格战可能会在短期内加剧，长期来看，拥有私有数据和专有场景的企业将获得更高的护城河。因此，企业在评估引入 AI 时，不仅要关注模型的参数大小，更要考量其背后的算力支持与持续维护成本。

## 冲击与重塑：职业前景与企业战略

关于"AI 会取代我的工作吗？”这一问题，目前的行业共识倾向于认为：**AI 职业发展规划**的重点应放在“增强”而非“替代”。虽然基础客服、初级翻译等重复性高、基于明确规则的任务确实面临被自动化的风险，但对于创意工作者、管理者和分析师而言，AI 更可能成为提升效率的强大杠杆。关键在于从业者能否从“执行者”转变为“指挥官”，学会利用工具来拓展自己的能力边界，而非固守旧有的工作流。

对于中小企业而言，**企业 AI 转型**的核心在于投资回报率（ROI）的评估。盲目跟风部署大型私有模型往往得不偿失。相反，通过集成成熟的 SaaS 工具来解决特定痛点，例如利用 AI 辅助营销文案生成或自动化客户支持，往往能以较低的成本实现显著的降本增效。根据 Gartner 的分析，成功实施 AI 项目的企业通常遵循“小步快跑”的原则，先在非核心业务环节进行试点，验证效果后再扩大规模，这种做法能有效降低试错风险。

我们需要客观地看到，尽管技术潜力巨大，但落地过程仍充满挑战。真实案例显示，某小型电商企业通过引入 AI 客服和选品分析系统，将运营成本降低了 30%，但这背后离不开对员工操作流程的重构和对数据的规范化管理。因此，技术本身并不是万能药，组织结构的适配性和人才的准备度才是决定 AI 成败的关键因素。未来的竞争格局，将更多地体现在谁能更快地将技术转化为实际的生产力，而非谁拥有了最先进的算法。

## 行动框架：从个人技能到组织规范

在个人层面，适应 AI 时代的第一步是更新技能树。掌握提示词工程（Prompt Engineering）已成为基本素养，它决定了你能从模型中挖掘出多少价值。此外，培养批判性思维至关重要，因为目前的 AI 仍存在“幻觉”问题，生成的内容并非绝对准确。我们建议读者建立“人机协作”的心态，将 AI 视为副驾驶，始终保留最终审核的权力。同时，保持对新技术的敏感度，定期关注行业动态，避免陷入知识停滞的状态。

对于企业决策者，推动**大模型落地**需要遵循严谨的步骤。首先，识别高价值的工作流程，确定哪些环节最适合引入 AI 辅助；其次，开展小范围试点（Pilot），收集真实数据以评估效果；最后，建立内部的数据规范与安全协议，防止敏感信息泄露。值得注意的是，不要盲目追求最大参数的模型，适合业务场景的中小型模型往往响应更快、成本更低且更易掌控。这一过程中，员工的培训与心态建设应与技术引入同步进行，以减少内部阻力。

在探索机会的同时，避坑清单同样重要。切勿忽视数据隐私合规风险，特别是在处理用户个人信息时，必须符合当地法律法规的要求。也不要过度迷信单一工具，保持工具链的开放性以便灵活切换。我们认为，建立一个包含技术团队、法务部门及业务骨干的跨职能小组，共同负责 AI 项目的评估与管理，是规避潜在风险的有效手段。只有在可控的前提下创新，企业才能确保持续的健康发展。

## 结语：拥抱杠杆，而非恐惧替代

回顾全文，我们可以清晰地看到，**人工智能趋势 2024** 已经从概念炒作走向了务实的应用阶段。无论是多模态能力的突破，还是智能体的兴起，都在指向一个更加智能化的未来。在这个过程中，焦虑是自然的反应，但恐慌无助于解决问题。AI 本质上是一个杠杆，它的存在是为了放大人类的智慧与创造力，而不是简单地抹除人类的价值。关键在于我们如何调整姿态，去善用这个杠杆撬动更大的成果。

正如业界广为流传的一句话所言：“未来的竞争不是人与 AI 的竞争，而是善用 AI 的人与不使用 AI 的人之间的竞争。”这句话或许略显简化，但它道出了主动适应的重要性。我们坚信，那些愿意拥抱变化、主动学习并合理部署 AI 的个人与组织，将在新一轮的经济周期中获得显著的先发优势。技术是中性的，赋予它意义的是使用它的人。

为了帮助您迈出第一步，我们整理了详细的《2024 个人 AI 工具包检查清单》，涵盖了从入门到进阶的实用资源与工具推荐。欢迎您在评论区留下目前最想解决的 AI 难题，我们将挑选典型问题进行后续的深度解答。如果您希望持续获取最新的 AI 实战周报与行业资讯，请订阅我们的通讯服务，让我们一起在这场技术变革中稳健前行。

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = await crew.kickoff_async(inputs={"topic": topic})

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).